In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
import logging
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- mane_mappings_filter_symbol ---
FIX_MANE_MAPPINGS_FILTER_SYMBOL_GENE_SYMBOL = "BRCA1"

# --- mane_mappings_genomic_filter ---
FIX_MANE_MAPPINGS_GENOMIC_FILTER_ALT_AC = "NC_000017.11"
FIX_MANE_MAPPINGS_GENOMIC_FILTER_END = 100
FIX_MANE_MAPPINGS_GENOMIC_FILTER_START = 0

# --- mane_mappings_isin ---

# --- mane_mappings_load ---
MANE_TSV_TEXT = """symbol	RefSeq_nuc	MANE_status	chr_start	chr_end	GRCh38_chr	name
BRCA1	NM_007294.4	MANE Select	41196312	41277500	NC_000017.11	BRCA1 DNA repair associated
TP53	NM_000546.6	MANE Select	7565097	7590868	NC_000017.11	tumor protein p53
EGFR	NM_005228.5	MANE Plus Clinical	55019017	55207338	NC_000007.14	epidermal growth factor receptor
"""

self = SimpleNamespace(
    df=pd.DataFrame({"symbol":["BRCA1","TP53","EGFR"],"RefSeq_nuc":["NM_007294.4","NM_000546.6","NM_005228.5"],"MANE_status":["MANE Select","MANE Select","MANE Plus Clinical"],"chr_start":[41196312,7565097,55019017],"chr_end":[41277500,7590868,55207338],"GRCh38_chr":["NC_000017.11","NC_000017.11","NC_000007.14"]}),
    mane_data_path=Path("/tmp/mane.tsv")
)
self.mane_data_path.write_text(MANE_TSV_TEXT, encoding="utf-8")

logger = logging.getLogger("mane-test")
logger.addHandler(logging.NullHandler())
DF_MANE_PD = self.df.copy()
DF_MANE_PL = pl.from_pandas(DF_MANE_PD)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_mane_mappings_filter_symbol(gene_symbol):
    data = self.df.loc[self.df["symbol"] == gene_symbol.upper()]

    if len(data) == 0:
        logger.warning(f"Unable to get MANE Transcript data for gene: {gene_symbol}")
        return None

    data = data.sort_values("MANE_status")
    return data.to_dict("records")
    return data

def before_mane_mappings_genomic_filter(alt_ac, end, start):
    mane_rows = self.df[
        (start >= self.df["chr_start"].astype(int))
        & (end <= self.df["chr_end"].astype(int))
        & (self.df["GRCh38_chr"] == alt_ac)
    ]
    mane_rows = mane_rows.sort_values("MANE_status", ascending=False)
    return mane_rows.to_dict("records")
    return mane_rows

def before_mane_mappings_isin():
    def get_mane_from_transcripts(self, transcripts: List[str]) -> List[Dict]:
        """Get mane transcripts from a list of transcripts

        :param List[str] transcripts: RefSeq transcripts on c. coordinate
        :return: MANE data
        """
        mane_rows = self.df["RefSeq_nuc"].isin(transcripts)
        result = self.df[mane_rows]
        if len(result) == 0:
            return []
        return result.to_dict("records")
    return get_mane_from_transcripts

def before_mane_mappings_load():
    return pd.read_csv(self.mane_data_path, delimiter="\t")
    return None

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_mane_mappings_filter_symbol(gene_symbol):
    data = self.df.filter(pl.col("symbol") == gene_symbol.upper())

    if len(data) == 0:
        logger.warning(f"Unable to get MANE Transcript data for gene: {gene_symbol}")
        return None

    data = data.sort("MANE_status")
    return data.to_dicts()
    return data

def gen_mane_mappings_genomic_filter(alt_ac, end, start):
    mane_rows = self.df.filter(
        (start >= self.df["chr_start"].cast(int))
        & (end <= self.df["chr_end"].cast(int))
        & (self.df["GRCh38_chr"] == alt_ac)
    )
    mane_rows = mane_rows.sort("MANE_status", descending=True)
    return mane_rows.to_dicts
    return mane_rows

def gen_mane_mappings_isin():
    def get_mane_from_transcripts(self, transcripts: List[str]) -> List[Dict]:
        """Get mane transcripts from a list of transcripts

        :param List[str] transcripts: RefSeq transcripts on c. coordinate
        :return: MANE data
        """
        mane_rows = self.df["RefSeq_nuc"].is_in(transcripts)
        result = self.df.filter(mane_rows)
        if len(result) == 0:
            return []
        return result.to_dicts()
    return get_mane_from_transcripts

def gen_mane_mappings_load():
    return pl.read_csv(self.mane_data_path, separator="\t")
    return None

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: mane_mappings_load ===

# L1 smoke – generated
try:
    _r = gen_mane_mappings_load()
    print("✅ L1 smoke gen_mane_mappings_load: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_mane_mappings_load: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_mane_mappings_load()
    print("✅ L1 smoke before_mane_mappings_load: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_mane_mappings_load: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_mane_mappings_load()
    _rg = gen_mane_mappings_load()
    compare(_rb, _rg, "mane_mappings_load")
except Exception as _e:
    print(f"❌ L2 equivalence mane_mappings_load: setup error — {type(_e).__name__}: {_e}")

# L3 edge - a valid header-only TSV preserves an empty schema.
import tempfile
try:
    _old_path = self.mane_data_path
    with tempfile.TemporaryDirectory() as _tmp:
        self.mane_data_path = Path(_tmp) / "empty_mane.tsv"
        self.mane_data_path.write_text(
            "symbol\tRefSeq_nuc\tMANE_status\tchr_start\tchr_end\tGRCh38_chr\tname\n",
            encoding="utf-8",
        )
        _rb = before_mane_mappings_load()
        _rg = gen_mane_mappings_load()
        compare(_rb, _rg, "L3 edge mane_mappings_load")
except Exception as _e:
    print(f"❌ L3 edge mane_mappings_load: {type(_e).__name__}: {_e}")
finally:
    self.mane_data_path = _old_path
